In [0]:
import json
import ast
import time
import random
import requests
import pandas as pd
from pydantic import BaseModel, field_validator
from typing import List, Optional, ClassVar, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed

class FacilityRecord(BaseModel):
    model_config = {"arbitrary_types_allowed": True}
    name: str
    facility_type: Optional[str] = None
    operator_type: Optional[str] = None
    phone_numbers: Optional[str] = None
    officialPhone: Optional[str] = None
    email: Optional[str] = None
    websites: Optional[str] = None
    officialWebsite: Optional[str] = None
    city: Optional[str] = None
    state: Optional[str] = None
    pincode: Optional[str] = None
    lat: Optional[float] = None
    lon: Optional[float] = None
    specialties: List[str] = []
    procedures: List[str] = []
    equipment: List[str] = []
    capabilities: List[str] = []
    number_doctors: Optional[int] = None
    year_established: Optional[int] = None
    n_followers: Optional[int] = None
    raw_notes: Optional[str] = None
    trust_score: Optional[float] = None
    trust_flags: List[str] = []
    reasoning: Optional[str] = None
    ai_reviewed: bool = False

    EMPTY_VALUES: ClassVar[tuple] = (
        "null", "nan", "none", "", "unknown", "n/a", "na", "-", "not available"
    )

    @field_validator("specialties", "procedures", "equipment", "capabilities", mode="before")
    @classmethod
    def parse_json_list(cls, v):
        if v is None: return []
        if isinstance(v, list): return v
        s = str(v).strip()
        if s.lower() in cls.EMPTY_VALUES or s == "[]": return []
        try:
            parsed = json.loads(s)
            return parsed if isinstance(parsed, list) else [str(parsed)]
        except:
            try:
                parsed = ast.literal_eval(s)
                return parsed if isinstance(parsed, list) else [str(parsed)]
            except:
                return [s]

    @field_validator("lat", "lon", mode="before")
    @classmethod
    def parse_decimal(cls, v):
        if v is None: return None
        try: return float(v)
        except: return None

    @field_validator("facility_type","operator_type","city","state","pincode",
                     "email","officialWebsite","officialPhone","websites","phone_numbers", mode="before")
    @classmethod
    def clean_null_strings(cls, v):
        if v is None or str(v).strip().lower() in cls.EMPTY_VALUES: return None
        return str(v).strip()

    @field_validator("pincode", mode="after")
    @classmethod
    def validate_pincode(cls, v):
        if v:
            clean = str(v).replace(".0", "").strip()
            return clean if (len(clean) == 6 and clean.isdigit()) else None
        return v

    @field_validator("number_doctors", "year_established", "n_followers", mode="before")
    @classmethod
    def parse_nullable_int(cls, v):
        if v is None or str(v).strip().lower() in cls.EMPTY_VALUES: return None
        try: return int(float(str(v)))
        except: return None

    @field_validator("trust_score", mode="after")
    @classmethod
    def score_range(cls, v):
        if v is not None and not (0.0 <= v <= 1.0):
            raise ValueError("trust_score must be between 0.0 and 1.0")
        return v

    @classmethod
    def from_row(cls, row: dict) -> "FacilityRecord":
        desc = str(row.get("description") or "")
        cap  = str(row.get("capability") or "")
        proc = str(row.get("procedure") or "")
        notes_parts = [p for p in [desc, cap, proc] if p.strip().lower() not in cls.EMPTY_VALUES]
        return cls(
            name             = str(row.get("name", "Unbekannt")),
            facility_type    = row.get("facilityTypeId"),
            operator_type    = row.get("operatorTypeId"),
            phone_numbers    = row.get("phone_numbers"),
            officialPhone    = row.get("officialPhone"),
            email            = row.get("email"),
            websites         = row.get("websites"),
            officialWebsite  = row.get("officialWebsite"),
            city             = row.get("address_city"),
            state            = row.get("address_stateOrRegion"),
            pincode          = str(row.get("address_zipOrPostcode", "")),
            lat              = row.get("latitude"),
            lon              = row.get("longitude"),
            specialties      = row.get("specialties"),
            procedures       = row.get("procedure"),
            equipment        = row.get("equipment"),
            capabilities     = row.get("capability"),
            number_doctors   = row.get("numberDoctors"),
            year_established = row.get("yearEstablished"),
            n_followers      = row.get("engagement_metrics_n_followers"),
            raw_notes        = " | ".join(notes_parts) if notes_parts else None,
        )

print("Schema geladen")

In [0]:
def _text(*fields) -> str:
    parts = []
    for f in fields:
        if isinstance(f, list): parts.extend([str(x).lower() for x in f if x])
        elif f: parts.append(str(f).lower())
    return " ".join(parts)

def _has(text: str, *keywords) -> bool:
    return any(kw in text for kw in keywords)

def compute_trust_score(record: FacilityRecord) -> Tuple[float, List[str]]:
    flags, score = [], 1.0
    all_text  = _text(record.raw_notes, record.specialties, record.procedures, record.capabilities)
    equip_text = _text(record.equipment)

    if _has(all_text, "surgery","surgical","appendectomy","operation","laparoscop","orthopedic","transplant","cardiac","neurosurgery") \
    and not _has(all_text + equip_text, "anesthes","anaesthes"):
        flags.append("⚠️ Surgery claimed, but no anesthetist found"); score -= 0.30

    if _has(all_text, "icu","intensive care","critical care","nicu","picu") \
    and not _has(equip_text + all_text, "ventilator","monitor","oxygen","defibrillator","ecg"):
        flags.append("⚠️ ICU claimed, but no equipment listed"); score -= 0.25

    if _has(all_text, "24/7","24 hour","always open","round the clock") \
    and _has(all_text, "part-time","part time","visiting doctor","limited hours"):
        flags.append("⚠️ 24/7 claimed, but part-time staff mentioned"); score -= 0.20

    if _has(_text(record.name, record.raw_notes), "advanced","multispecial","super special","state-of-the-art") \
    and len(record.specialties) < 2:
        flags.append("⚠️ 'Advanced' claimed, but few specialties listed"); score -= 0.20

    if len(record.specialties) > 3 and len(record.raw_notes or "") < 50:
        flags.append("⚠️ Many specialties, but little description text"); score -= 0.15

    if _has(all_text, "emergency","trauma","casualty","urgent care") \
    and not _has(all_text, "24/7","24 hour","always open","ambulance"):
        flags.append("⚠️ Emergency mentioned, but no 24/7 indication"); score -= 0.10

    if not record.lat or not record.lon:
        flags.append("ℹ️ No GPS coordinates"); score -= 0.05
    if not record.websites and not record.officialWebsite:
        flags.append("ℹ️ No website listed"); score -= 0.05
    if not record.email:
        flags.append("ℹ️ No email listed"); score -= 0.05
    if not record.phone_numbers and not record.officialPhone:
        flags.append("ℹ️ No phone number listed"); score -= 0.05

    return round(max(score, 0.0), 2), flags

print("Trust Scorer geladen")

In [0]:
from openai import OpenAI

token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
host  = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()

client = OpenAI(
    api_key=token,
    base_url=f"{host}/serving-endpoints"
)

def ai_trust_review_batch(batch: list) -> list:
    records_text = ""
    for idx, r in enumerate(batch):
        records_text += f"""
RECORD {idx}:
  Name: {r.name} | Type: {r.facility_type} | State: {r.state}
  Notes: {(r.raw_notes or "")[:200]}
  Specialties: {', '.join(r.specialties[:5]) if r.specialties else "None"}
  Equipment: {', '.join(r.equipment[:5]) if r.equipment else "None"}
  Flags: {', '.join(r.trust_flags) if r.trust_flags else "None"}
"""

    prompt = f"""You are a medical quality auditor. Review these {len(batch)} Indian healthcare facilities.

{records_text}

Respond ONLY with a JSON array with exactly {len(batch)} objects, one per record in order:
[
  {{"trust_score": 0.85, "confirmed_flags": [], "dismissed_flags": [], "new_flags": [], "reasoning": "one sentence"}},
  ...
]"""

    response = client.chat.completions.create(
        model="databricks-meta-llama-3-1-8b-instruct",
        messages=[{"role": "user", "content": prompt}]
    )
    raw   = response.choices[0].message.content
    start = raw.find("[")
    end   = raw.rfind("]") + 1
    return json.loads(raw[start:end])

print("KI-Review Batch-Funktion geladen")

In [0]:
rows    = spark.table("Base_table").collect()
records = [FacilityRecord.from_row(row.asDict() if hasattr(row, 'asDict') else dict(row)) for row in rows]
print(f"{len(records)} Records erstellt")

In [0]:
from openai import OpenAI
import json, time, re

token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
host  = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
client = OpenAI(api_key=token, base_url=f"{host}/serving-endpoints")

def ai_trust_review_batch(batch: list) -> list:
    records_text = ""
    for idx, r in enumerate(batch):
        records_text += f"\nRECORD {idx}: Name:{r.name} | Type:{r.facility_type} | State:{r.state} | Notes:{(r.raw_notes or '')[:100]} | Flags:{', '.join(r.trust_flags) if r.trust_flags else 'None'}"

    prompt = f"""You are a medical quality auditor. Review these {len(batch)} Indian healthcare facilities.
{records_text}

Respond ONLY with a valid JSON array of exactly {len(batch)} objects:
[{{"trust_score":0.85,"confirmed_flags":[],"new_flags":[],"reasoning":"max 8 words"}}]
No newlines in strings. No trailing commas."""

    response = client.chat.completions.create(
        model="databricks-meta-llama-3-1-8b-instruct",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1,
        max_tokens=600
    )
    raw = response.choices[0].message.content.strip()
    raw = re.sub(r',\s*]', ']', raw)
    raw = re.sub(r',\s*}', '}', raw)
    start = raw.find("["); end = raw.rfind("]") + 1
    return json.loads(raw[start:end])

for record in records:
    score, flags = compute_trust_score(record)
    record.trust_score = score
    record.trust_flags = flags
    record.reasoning   = "Rule-based score"
    record.ai_reviewed = False

needs_review = sorted(
    [r for r in records if any("⚠️" in f for f in r.trust_flags)],
    key=lambda r: r.trust_score
)[:100]

print(f"{len(records)} Records regelbasiert gescort")
print(f"KI-Review für Top 100 kritischste Records")

BATCH_SIZE = 5
success = 0

for i in range(0, len(needs_review), BATCH_SIZE):
    batch = needs_review[i:i+BATCH_SIZE]
    for attempt in range(3):
        try:
            results = ai_trust_review_batch(batch)
            for record, ai_result in zip(batch, results):
                record.trust_score = ai_result["trust_score"]
                record.trust_flags = ai_result.get("confirmed_flags", []) + ai_result.get("new_flags", [])
                record.reasoning   = ai_result["reasoning"]
                record.ai_reviewed = True
                success += 1
            time.sleep(3)
            break
        except Exception as e:
            if "429" in str(e):
                time.sleep(60)
            else:
                print(f"   FEHLER Batch {i}: {type(e).__name__}: {e}")
                time.sleep(5)
    print(f"   Batch {i+BATCH_SIZE}/100 done")

print(f"\nKI-Review: {success}/100 erfolgreich")
print(f"🟢 >= 0.8 : {sum(1 for r in records if r.trust_score >= 0.8)}")
print(f"🟡 0.5–0.8: {sum(1 for r in records if 0.5 <= r.trust_score < 0.8)}")
print(f"🔴 < 0.5  : {sum(1 for r in records if r.trust_score < 0.5)}")
print(f"AI reviewed: {sum(1 for r in records if r.ai_reviewed)}")

In [0]:
df_scored = pd.DataFrame([r.model_dump() for r in records])

spark.createDataFrame(df_scored) \
     .write.format("delta") \
     .mode("overwrite") \
     .saveAsTable("facilities_scored")

print(f"✅ Gespeichert als Delta table: facilities_scored")
print(f"🟢 Score >= 0.8 : {len(df_scored[df_scored.trust_score >= 0.8])}")
print(f"🟡 Score 0.5–0.8: {len(df_scored[(df_scored.trust_score >= 0.5) & (df_scored.trust_score < 0.8)])}")
print(f"🔴 Score < 0.5  : {len(df_scored[df_scored.trust_score < 0.5])}")